# Delta-Forward Scattering and Angular Operators

Delta-forward scattering preserves direction, so $\Sigma_{s,\ell}=\Sigma_s$ for every Legendre moment. The transport equation reduces to

$$\boldsymbol\Omega\!\cdot\!\nabla\psi+(\Sigma_t-\Sigma_s)\psi=Q.$$

This pure-absorber equivalence provides a reference for comparing the standard, Galerkin One, and Galerkin Three angular operators.

In [ ]:
from pathlib import Path

import numpy as np
from mpi4py import MPI

from pyopensn.aquad import GLCProductQuadrature3DXYZ
from pyopensn.context import Finalize, UseColor
from pyopensn.fieldfunc import FieldFunctionInterpolationVolume
from pyopensn.logvol import RPPLogicalVolume
from pyopensn.mesh import OrthogonalMeshGenerator
from pyopensn.solver import DiscreteOrdinatesProblem, SteadyStateSourceSolver
from pyopensn.source import VolumetricSource
from pyopensn.xs import MultiGroupXS

UseColor(False)
comm = MPI.COMM_WORLD
rank = comm.rank

## Problem setup

| Parameter | Value |
|---|---|
| Geometry | $[0,10]^3$ with $20^3$ cells |
| Cross sections | $\Sigma_t=1.5$, $\Sigma_{s,\ell}=1.0$, $\Sigma_a=0.5$ |
| Source | Uniform isotropic source, $Q=1.0$ |
| Boundaries | Isotropic incoming flux $\psi_b=2.0$ |
| Quadrature | GL-Chebyshev, $N_p=4$, $N_\phi=8$ (32 directions) |
| Reference | Uniform scalar flux $\phi^*=Q/\Sigma_a=2.0$ |

Standard P7 retains 64 moments. Galerkin One and Galerkin Three each select 32 independent moments for the 32 directions.

In [ ]:
sigma_t = 1.5
sigma_s = 1.0
sigma_a = sigma_t - sigma_s
source_strength = 1.0
reference_flux = source_strength / sigma_a
scattering_order = 7

num_cells = 20
domain_length = 10.0
nodes = [domain_length * i / num_cells for i in range(num_cells + 1)]
mesh = OrthogonalMeshGenerator(node_sets=[nodes, nodes, nodes]).Execute()
mesh.SetUniformBlockID(0)

xs_filename = Path("delta_forward.xs")
reference_xs_filename = Path("pure_absorber.xs")
if rank == 0:
    with xs_filename.open("w") as stream:
        stream.write("NUM_GROUPS 1\n")
        stream.write(f"NUM_MOMENTS {scattering_order + 1}\n\n")
        stream.write("SIGMA_T_BEGIN\n")
        stream.write(f"0 {sigma_t}\n")
        stream.write("SIGMA_T_END\n\n")
        stream.write("TRANSFER_MOMENTS_BEGIN\n")
        for ell in range(scattering_order + 1):
            stream.write(f"M_GFROM_GTO_VAL {ell} 0 0 {sigma_s}\n")
        stream.write("TRANSFER_MOMENTS_END\n")

    with reference_xs_filename.open("w") as stream:
        stream.write("NUM_GROUPS 1\n")
        stream.write("NUM_MOMENTS 1\n\n")
        stream.write("SIGMA_T_BEGIN\n")
        stream.write(f"0 {sigma_a}\n")
        stream.write("SIGMA_T_END\n\n")
        stream.write("TRANSFER_MOMENTS_BEGIN\n")
        stream.write("TRANSFER_MOMENTS_END\n")
comm.Barrier()

## Check the angular operators

The product of the transposed stored discrete-to-moment array and the moment-to-discrete array should equal the identity in moment space. We report

- the largest off-diagonal magnitude, and
- the largest diagonal deviation from one.

Both metrics are zero for consistent transforms.

In [ ]:
def operator_errors(quadrature):
    d2m_transpose = quadrature.GetDiscreteToMomentOperator()
    m2d = quadrature.GetMomentToDiscreteOperator()
    product = d2m_transpose.T @ m2d
    diagonal = np.diag(product)
    off_diagonal = ~np.eye(*product.shape, dtype=bool)
    max_off_diagonal = np.abs(product[off_diagonal]).max()
    max_diagonal_deviation = np.abs(diagonal - 1.0).max()
    return max_off_diagonal, max_diagonal_deviation

quadratures = {
    "STANDARD": GLCProductQuadrature3DXYZ(
        n_polar=4, n_azimuthal=8, scattering_order=7,
        operator_method="standard"
    ),
    "GALERKIN_ONE": GLCProductQuadrature3DXYZ(
        n_polar=4, n_azimuthal=8, operator_method="galerkin_one"
    ),
    "GALERKIN_THREE": GLCProductQuadrature3DXYZ(
        n_polar=4, n_azimuthal=8, scattering_order=7,
        operator_method="galerkin_three"
    ),
}

operator_results = {
    name: operator_errors(quadrature)
    for name, quadrature in quadratures.items()
}
if rank == 0:
    print(f"{'Method':<20} {'Max off-diagonal':>20} {'Max diag deviation':>20}")
    print("-" * 62)
    for name, (off_diagonal, diagonal_deviation) in operator_results.items():
        label = name.replace("_", " ").title()
        print(f"{label:<20} {off_diagonal:>20.6e} {diagonal_deviation:>20.6e}")
        print(f"{name}_DM_MAX_OFF_DIAGONAL={off_diagonal:.12e}")
        print(f"{name}_DM_MAX_DIAG_DEVIATION={diagonal_deviation:.12e}")

| Method | Maximum off-diagonal | Maximum diagonal deviation |
|---|---:|---:|
| Standard P7 | 2.335 | 1.559 |
| Galerkin One | $2.96\times10^{-15}$ | $1.44\times10^{-15}$ |
| Galerkin Three | $2.88\times10^{-16}$ | $1.11\times10^{-15}$ |

The standard mapping is inconsistent for this moment-direction pairing; both Galerkin mappings agree with the identity to roundoff.

## Compare transport solutions

Each solve is compared with $\phi^*=2$ using two response metrics:

$$E_{avg}=\frac{|\bar\phi-\phi^*|}{\phi^*},\qquad E_{peak}=\frac{\phi_{max}-\bar\phi}{\phi^*}.$$

$E_{avg}$ measures bias in the domain average; $E_{peak}$ measures departure from the uniform reference.

In [ ]:
def run_transport(quadrature, cross_section_file):
    cross_sections = MultiGroupXS()
    cross_sections.LoadFromOpenSn(str(cross_section_file))
    source = VolumetricSource(
        block_ids=[0], group_strength=[source_strength]
    )
    problem = DiscreteOrdinatesProblem(
        mesh=mesh,
        num_groups=1,
        groupsets=[{
            "groups_from_to": (0, 0),
            "angular_quadrature": quadrature,
            "inner_linear_method": "petsc_gmres",
            "l_abs_tol": 1.0e-8,
            "l_max_its": 100,
            "gmres_restart_interval": 100,
        }],
        xs_map=[{"block_ids": [0], "xs": cross_sections}],
        volumetric_sources=[source],
        boundary_conditions=[
            {"name": name, "type": "isotropic",
             "group_strength": [reference_flux]}
            for name in ("xmin", "xmax", "ymin", "ymax", "zmin", "zmax")
        ],
        options={"verbose_inner_iterations": False},
    )
    solver = SteadyStateSourceSolver(problem=problem)
    solver.Initialize()
    solver.Execute()

    scalar_flux = problem.GetScalarFluxFieldFunction()[0]
    domain = RPPLogicalVolume(infx=True, infy=True, infz=True)
    values = {}
    for operation in ("max", "avg"):
        interpolation = FieldFunctionInterpolationVolume()
        interpolation.SetOperationType(operation)
        interpolation.SetLogicalVolume(domain)
        interpolation.AddFieldFunction(scalar_flux)
        interpolation.Execute()
        values[operation] = interpolation.GetValue()
    return values["max"], values["avg"]

reference_quadrature = GLCProductQuadrature3DXYZ(
    n_polar=4, n_azimuthal=8, scattering_order=0
)
flux_results = {
    "REFERENCE": run_transport(reference_quadrature, reference_xs_filename)
}
for name, quadrature in quadratures.items():
    flux_results[name] = run_transport(quadrature, xs_filename)

In [ ]:
response_metrics = {}
for name, (maximum, average) in flux_results.items():
    average_error = abs(average - reference_flux) / reference_flux
    peak_deviation = (maximum - average) / reference_flux
    response_metrics[name] = (average_error, peak_deviation)

if rank == 0:
    print(f"{'Case':<20} {'Max flux':>12} {'Avg flux':>12} "
          f"{'E_avg':>12} {'E_peak':>12}")
    print("-" * 72)
    for name, (maximum, average) in flux_results.items():
        average_error, peak_deviation = response_metrics[name]
        label = name.replace("_", " ").title()
        print(f"{label:<20} {maximum:>12.6e} {average:>12.6e} "
              f"{average_error:>12.6e} {peak_deviation:>12.6e}")
        print(f"{name}_AVERAGE_RELATIVE_ERROR={average_error:.12e}")
        print(f"{name}_NORMALIZED_PEAK_DEVIATION={peak_deviation:.12e}")

## Interpret the results

| Case | Average flux | $E_{avg}$ | $E_{peak}$ | Solver status |
|---|---:|---:|---:|---|
| Pure-absorber reference | 2.00000 | 0 | $6.66\times10^{-16}$ | Converged |
| Standard P7 | 2.02466 | $1.23\times10^{-2}$ | $1.35\times10^{-1}$ | Iteration limit |
| Galerkin One | 2.00000 | $4.33\times10^{-12}$ | $8.45\times10^{-10}$ | Converged |
| Galerkin Three | 2.00000 | $1.47\times10^{-12}$ | $8.34\times10^{-10}$ | Converged |

Standard P7 maps 64 moments through only 32 directions, so $DM$ is not the identity and its source iteration stalls. Its flux values are diagnostic, not a converged error estimate. Both Galerkin methods select a consistent 32-moment space and recover the uniform reference to numerical precision.

In [ ]:
if rank == 0:
    xs_filename.unlink(missing_ok=True)
    reference_xs_filename.unlink(missing_ok=True)

## Finalize (for Jupyter Notebook only)

In script mode, PyOpenSn handles finalization automatically. In a Jupyter kernel, finalize OpenSn before MPI.

In [ ]:
if "opensn_console" not in globals():
    from IPython import get_ipython

    if get_ipython() is not None:
        Finalize()
        MPI.Finalize()